# 01. 데이터 인벤토리 — MIMIC-III-Ext-PPG

**목적**: 보유 데이터의 실제 규모·구성·라벨 지형을 확인하고, 이후 모든 분석이 참조할
세그먼트 인덱스(`seg_index.parquet`)를 만든다.

## 데이터셋 개요
- 30초 PPG 세그먼트, 125 Hz, WFDB 포맷 (`p00`~`p09` 분할)
- 세그먼트당 채널: PLETH(필수), ECG II, RESP, ABP(선택)
- 메타데이터 39개 컬럼: 리듬, SQI(pleth/ecg/abp/resp), SBP/DBP/HR/RR, 인구통계, ICD-9/10, strat_fold

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
from ppg_fm.config import load, ds

CFG  = load()
DATA = CFG["datasets"]["mimic_ext_ppg"]["root"]
INT  = os.path.join(ROOT, "data", "interim")
REP  = os.path.join(ROOT, "reports")
os.makedirs(INT, exist_ok=True); os.makedirs(REP, exist_ok=True)
print("data :", DATA)

## 1. 파형 파일 구조 확인

헤더(`.hea`)를 직접 읽어 채널 구성과 샘플링을 확인한다.

In [ ]:
import glob, random, collections
random.seed(0)

pats = [l.strip() for l in open(os.path.join(DATA, "RECORDS")) if l.strip()]
print(f"환자 디렉터리: {len(pats):,}")

# 헤더 예시
h = glob.glob(os.path.join(DATA, pats[0], "*.hea"))[0]
print("\n--- 헤더 예시 ---")
print(open(h).read())

### 채널 조합 분포

어떤 채널이 얼마나 자주 동반되는지 — 특히 **ABP 동반율**이 혈역학 분석의 상한을 결정한다.

In [ ]:
combo, any_ch = collections.Counter(), collections.Counter()
for p in random.sample(pats, 120):
    heas = glob.glob(os.path.join(DATA, p, "*.hea"))
    for hf in random.sample(heas, min(8, len(heas))):
        lines = [l.strip() for l in open(hf) if l.strip()]
        chans = tuple(l.split()[-1] for l in lines[1:])
        combo[chans] += 1
        for c in chans: any_ch[c] += 1

tot = sum(combo.values())
print("=== 채널 조합 ===")
for k, v in combo.most_common(8):
    print(f"  {str(k):42s} {100*v/tot:5.1f}%")
print("\n=== 채널별 출현율 ===")
for k, v in any_ch.most_common():
    print(f"  {k:8s} {100*v/tot:5.1f}%")

## 2. 신호 품질 실측

실제 파형을 로드해 NaN·flatline·값 범위를 확인한다.

In [ ]:
import wfdb

stats = []
for p in random.sample(pats, 40):
    heas = glob.glob(os.path.join(DATA, p, "*.hea"))
    for hf in random.sample(heas, min(3, len(heas))):
        try:
            r = wfdb.rdrecord(hf[:-4])
            x = r.p_signal[:, r.sig_name.index("PLETH")]
            stats.append(dict(n=len(x), fs=r.fs, nan=np.isnan(x).mean(),
                              lo=np.nanmin(x), hi=np.nanmax(x), sd=np.nanstd(x)))
        except Exception:
            pass

S = pd.DataFrame(stats)
print(f"로드 성공 {len(S)}건")
print(f"  길이      : {sorted(S.n.unique())} 샘플")
print(f"  샘플링    : {sorted(S.fs.unique())} Hz")
print(f"  NaN 비율  : 중앙값 {S.nan.median():.3%}  최대 {S.nan.max():.2%}")
print(f"  값 범위   : {S.lo.min():.3f} ~ {S.hi.max():.3f}")
print(f"  flatline  : {(S.sd < 1e-6).sum()} / {len(S)}")

## 3. 세그먼트 인덱스 구축

메타데이터 640만 행을 읽어 이후 분석의 기준이 되는 인덱스를 만든다.

**품질 정의**: `vector_10s_pleth_sqi == [1,1,1]` — 30초를 3등분한 10초 구간이 **모두** 양호.
**질환 라벨**: `icd10_truncated`(3자리 코드 리스트)를 문자열 매칭으로 플래그화.

In [ ]:
COLS = ["folder_path","subject_id","event_rhythm","age","gender",
        "vector_10s_pleth_sqi","vector_10s_abp_sqi","icd10_truncated","strat_fold"]
raw = pd.read_csv(os.path.join(DATA, "metadata.csv"), usecols=COLS, dtype=str)
print(f"메타데이터 {len(raw):,} 행")

hq  = raw["vector_10s_pleth_sqi"].fillna("").str.replace(" ", "", regex=False).eq("[1,1,1]")
abp = ~raw["vector_10s_abp_sqi"].fillna("nan").str.contains("nan")
icd = raw["icd10_truncated"].fillna("")

CODES = ["I35","I34","I50","I48","I25","I21","I95","R57"]
idx = pd.DataFrame({
    "folder_path": raw.folder_path,
    "subject":     raw.subject_id,
    "rhythm":      raw.event_rhythm,
    "age":         pd.to_numeric(raw.age, errors="coerce"),
    "gender":      raw.gender,
    "fold":        raw.strat_fold,
    "hq":          hq,
    "has_abp":     abp,
    **{c: icd.str.contains(f"'{c}'") for c in CODES},
})
idx["cardiac"] = idx[["I35","I34","I50","I48","I25","I21"]].any(axis=1)
idx.to_parquet(os.path.join(INT, "seg_index.parquet"), index=False)

print(f"고품질      : {idx.hq.sum():,} ({100*idx.hq.mean():.1f}%)")
print(f"ABP 동반    : {idx.has_abp.sum():,} ({100*idx.has_abp.mean():.1f}%)")
print(f"고품질∩ABP  : {(idx.hq & idx.has_abp).sum():,}")

## 4. 질환 라벨 지형

환자 단위로 ICD-10 코드를 집계한다. **이 규모가 phenome-wide 분석의 상한**이다.

In [ ]:
import ast

pat_codes = {}
for row in pd.read_csv(os.path.join(DATA, "metadata.csv"),
                       usecols=["subject_id","icd10_truncated"], dtype=str).itertuples():
    if row.subject_id in pat_codes: continue
    try:    pat_codes[row.subject_id] = set(ast.literal_eval(row.icd10_truncated))
    except Exception: pat_codes[row.subject_id] = set()

cnt = collections.Counter()
for codes in pat_codes.values():
    for c in codes: cnt[c] += 1

N = len(pat_codes)
print(f"환자 {N:,} / 고유 ICD-10 코드 {len(cnt):,}")
for thr in (200, 100, 50, 20):
    print(f"  환자 >={thr:3d}명 코드 : {sum(1 for v in cnt.values() if v >= thr):4d}개")

census = pd.DataFrame(sorted(cnt.items(), key=lambda x: -x[1]),
                      columns=["icd10","n_patients"])
census["pct"] = (100*census.n_patients/N).round(2)
census.to_csv(os.path.join(REP, "icd_census.csv"), index=False)
census.head(25)

### 심혈관 관련 코드

특히 **I35(대동맥판막)·I34(승모판)** 이 PPG 파형과 함께 존재하는지가 핵심.

In [ ]:
CARDIO = {"I35":"대동맥판막 장애","I34":"승모판 장애","I50":"심부전",
          "I48":"심방세동·조동","I25":"만성 허혈성심질환","I21":"급성 심근경색",
          "I95":"저혈압","I27":"폐성심·폐순환","R57":"쇼크","I46":"심정지","I26":"폐색전증"}
for c, ko in CARDIO.items():
    v = cnt.get(c, 0)
    print(f"  {c:5s} {ko:16s} {v:5,}명 ({100*v/N:5.1f}%)")

## 정리

| 항목 | 값 |
|---|---|
| 총 세그먼트 | 6,399,754 |
| 고품질 (SQI 3/3) | 5,401,385 (84.4%) |
| ABP 동반 | 2,957,965 (46.2%) |
| 환자 | 6,189 |
| ICD-10 코드 (≥100명) | 174개 |

→ 다음: `02_morphology_features.ipynb` (특징 추출기 검증)